In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score
sys.path.append(str(Path("..")))
from src.dataset import train_data, train_loader, val_loader, test_loader,test_path
from torchvision import transforms,datasets
from torch.utils.data import Subset,DataLoader

torch.Size([32, 3, 224, 224])
torch.Size([32])
tensor([2, 2, 5, 6, 6, 0, 4, 3, 5, 0])


In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

cuda
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


# *StepLR*

In [3]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [4]:
num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)


In [5]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=30,
    gamma=0.1
)

In [6]:
num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []


In [7]:
for epoch in range(num_epochs):

    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total


    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average=None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    val_f1_scores.append(val_f1)
    val_f1_scores_by_class.append(f1_score_by_class)


    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    print("F1_Score by each class")
    print(f1_score_by_class)

    print("=======================================================================================")


    scheduler.step()

Epoch [1/100] | LR: 0.001000 | Train Loss: 1.4613 | Train Acc: 0.4959 | Val Loss: 0.8677 | Val Acc: 0.6802 | Val Macro F1: 0.6733
F1_Score by each class
[0.66666667 0.75641026 0.5477707  0.3968254  0.70588235 0.79807692
 0.8994709  0.61538462]
Epoch [2/100] | LR: 0.001000 | Train Loss: 0.6374 | Train Acc: 0.7665 | Val Loss: 0.8592 | Val Acc: 0.7087 | Val Macro F1: 0.6777
F1_Score by each class
[0.74418605 0.86597938 0.47761194 0.52054795 0.67272727 0.73333333
 0.90710383 0.5       ]
Epoch [3/100] | LR: 0.001000 | Train Loss: 0.3741 | Train Acc: 0.8667 | Val Loss: 0.5578 | Val Acc: 0.8168 | Val Macro F1: 0.8005
F1_Score by each class
[0.80291971 0.91578947 0.74257426 0.63636364 0.8375     0.86829268
 0.95431472 0.64615385]
Epoch [4/100] | LR: 0.001000 | Train Loss: 0.2061 | Train Acc: 0.9294 | Val Loss: 0.7116 | Val Acc: 0.8063 | Val Macro F1: 0.7828
F1_Score by each class
[0.83333333 0.91099476 0.69747899 0.59060403 0.85365854 0.83902439
 0.95918367 0.57777778]
Epoch [5/100] | LR: 0.00

In [8]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7781
Test Macro F1: 0.7740
Test F1 by class:
[0.8        0.9        0.65306122 0.6875     0.83495146 0.76785714
 0.93877551 0.6097561 ]


# *ReduceLR*

In [8]:
class CNN(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [9]:
num_classes = len(train_data.classes)

model = CNN(num_classes)
model = model.to(device)


In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=5
)

In [11]:
num_epochs = 100

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
val_f1_scores = []
val_f1_scores_by_class = []


In [12]:
for epoch in range(num_epochs):

    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = running_train_loss / len(train_loader)
    train_accuracy = train_correct / train_total


    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    all_val_labels = []
    all_val_predictions = []

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

            all_val_labels.extend(labels.cpu().numpy())
            all_val_predictions.extend(predicted.cpu().numpy())

    val_loss = running_val_loss / len(val_loader)
    val_accuracy = val_correct / val_total

    val_f1 = f1_score(
        all_val_labels,
        all_val_predictions,
        average="macro"
    )

    f1_score_by_class = f1_score(
        all_val_labels,
        all_val_predictions,
        average=None
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    val_f1_scores.append(val_f1)
    val_f1_scores_by_class.append(f1_score_by_class)

    
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step(val_loss)
    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"LR: {current_lr:.6f} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Val Macro F1: {val_f1:.4f}"
    )

    print("F1_Score by each class")
    print(f1_score_by_class)

    print("=======================================================================================")
    


Epoch [1/100] | LR: 0.001000 | Train Loss: 1.6363 | Train Acc: 0.4313 | Val Loss: 1.0394 | Val Acc: 0.6502 | Val Macro F1: 0.6242
F1_Score by each class
[0.62903226 0.73333333 0.53881279 0.56410256 0.60273973 0.73429952
 0.78125    0.41025641]
Epoch [2/100] | LR: 0.001000 | Train Loss: 0.7717 | Train Acc: 0.7252 | Val Loss: 0.7973 | Val Acc: 0.7282 | Val Macro F1: 0.7011
F1_Score by each class
[0.72058824 0.84571429 0.60227273 0.66298343 0.74444444 0.74774775
 0.80909091 0.47619048]
Epoch [3/100] | LR: 0.001000 | Train Loss: 0.4693 | Train Acc: 0.8356 | Val Loss: 0.7111 | Val Acc: 0.7583 | Val Macro F1: 0.7380
F1_Score by each class
[0.68468468 0.87292818 0.67592593 0.56603774 0.77906977 0.79497908
 0.93684211 0.59375   ]
Epoch [4/100] | LR: 0.001000 | Train Loss: 0.2796 | Train Acc: 0.8968 | Val Loss: 0.6292 | Val Acc: 0.7883 | Val Macro F1: 0.7697
F1_Score by each class
[0.80745342 0.90052356 0.67       0.63953488 0.81927711 0.84878049
 0.89010989 0.58181818]
Epoch [5/100] | LR: 0.00

In [10]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_data = datasets.ImageFolder(
    test_path,
    transform=test_transform
)

test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False
)


model.eval()

all_test_labels = []
all_test_predictions = []

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

        all_test_labels.extend(labels.cpu().numpy())
        all_test_predictions.extend(predicted.cpu().numpy())
        
test_accuracy = test_correct / test_total
test_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)
test_f1_by_class = f1_score(
    all_test_labels,
    all_test_predictions,
    average=None
)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Macro F1: {test_f1:.4f}")

print("Test F1 by class:")
print(test_f1_by_class)


Test Accuracy: 0.7781
Test Macro F1: 0.7747
Test F1 by class:
[0.83870968 0.88421053 0.65384615 0.59813084 0.9        0.80701754
 0.94845361 0.56756757]
